# E8c · k=1/2/4/8 + отдельные отчёты о цене

**Без обучения:** проверяем настоящие состояния B5 из середины сегментов stage 3. B0, B5 и адаптер E9 не меняются.

Открой в Colab с GPU, выполняй ячейки сверху вниз. Готовый E8-кэш восстановится из Drive. Путь к архивам — `datasets/cityscapes`. Новые результаты — `Мой диск/e8_cityscapes_v1/e8c_k_sweep_v1/`; предыдущие E8/E8b/E9 и первый E8c не перезаписываются.

**Уже измеренную цену из `e8c_cost_v1` повторно не считаем**, если сохранены полный отчёт и исходный скрипт, совпадают версия кода, GPU, PyTorch/CUDA и режим TF32. В отчёте будет отметка, что это прежние измерения. Если условия не совпали — запускается новый профиль. Расчёт сценариев обновляется для k=1/2/4/8, включая k контекстов плюс отдельный final.

Результаты качества остаются teacher-assisted: настоящий вход выбранных токенов и ранние карты B5, настоящий final вне регионов. Это диагностическая проверка расписания, не строгая верхняя граница для обучаемого адаптера и не измерение разреженного ускорения. ACDC пока не используется.


In [ ]:
# 1. Окружение. PyTorch/CUDA не переустанавливаем.
import os, sys, subprocess
os.environ['USE_TF'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.46.3', 'cityscapesscripts==2.2.4', 'scikit-learn', 'tqdm', 'fvcore==0.1.5.post20221221'], check=True)
import torch
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения → Сменить среду выполнения'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Укажи папку с архивами. Обычно нужно поменять ТОЛЬКО эту строку.
from pathlib import Path
ARCHIVE_DIR = Path('/content/drive/MyDrive/datasets/cityscapes')  # <-- твоя папка или ярлык
IMAGE_ZIP = ARCHIVE_DIR / 'val.zip'
GT_ZIP = ARCHIVE_DIR / 'gtFine_trainvaltest.zip'

# Если архив называется иначе, поменяй IMAGE_ZIP/GT_ZIP выше.
for p in [IMAGE_ZIP, GT_ZIP]:
    assert p.is_file(), f'Не найден {p}. Найди архив в панели Files → drive → MyDrive и скопируй путь.'

RUN_NAME = 'e8_cityscapes_v1'  # для нового независимого запуска используй другое имя
SAVE = Path('/content/drive/MyDrive') / RUN_NAME
WORK = Path('/content') / RUN_NAME
EXP = WORK / 'experiments'
DATA = WORK / 'data'
CACHE = WORK / 'cache'
RESULTS = SAVE / 'results'
for p in [SAVE, EXP, DATA, CACHE, RESULTS]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(E8_DATA=str(DATA), E8_CACHE=str(CACHE), E8_RESULTS=str(RESULTS), E8_BACKUP=str(SAVE/'backups'))
print('Результаты:', SAVE)


## Распаковка только validation
Поддерживаются `val/<город>/...` и `leftImg8bit/val/<город>/...` в архиве изображений. Из разметки берём только `gtFine/val/*/*_labelIds.png`. Полный train/test не распаковывается.

In [ ]:
# 3. Извлечение и проверка 500 пар.
import zipfile, shutil
from collections import Counter

def unpack_cityscapes(archive, image):
    count = 0
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            parts = Path(member.filename).parts
            suffix = '_leftImg8bit.png' if image else '_gtFine_labelIds.png'
            if member.is_dir() or not member.filename.endswith(suffix) or 'val' not in parts:
                continue
            k = parts.index('val')
            tail = parts[k:]
            if len(tail) != 3 or '..' in tail: continue
            branch = 'leftImg8bit' if image else 'gtFine'
            out = DATA/'cityscapes'/branch/Path(*tail)
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != member.file_size:
                tmp = out.with_suffix(out.suffix+'.part')
                with z.open(member) as src, tmp.open('wb') as dst: shutil.copyfileobj(src,dst)
                tmp.replace(out)
            count += 1
    return count

print('Изображений:', unpack_cityscapes(IMAGE_ZIP, True))
print('Масок:', unpack_cityscapes(GT_ZIP, False))
images = sorted((DATA/'cityscapes/leftImg8bit/val').glob('*/*_leftImg8bit.png'))
counts = Counter(p.parent.name for p in images)
assert dict(counts) == {'frankfurt':267, 'lindau':59, 'munster':174}, counts
for p in images:
    gt = DATA/'cityscapes/gtFine/val'/p.parent.name/p.name.replace('_leftImg8bit.png','_gtFine_labelIds.png')
    assert gt.is_file(), f'Отсутствует маска {gt}'
print('500 пар OK:', dict(counts))


In [ ]:
# 4. Создание автономных скриптов (никакие файлы репозитория не меняются).
import json, hashlib
SOURCES = {'common.py': 'import os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom transformers import SegformerForSemanticSegmentation\nfrom cityscapesscripts.helpers.labels import labels\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ["E8_DATA"])\nCACHE = Path(os.environ["E8_CACHE"])\nRESULTS = Path(os.environ["E8_RESULTS"])\n\nSIZE = (512, 1024)\nNUM_CLASSES = 19\nIGNORE = 255\n\nMODELS = {\n    "b0": "nvidia/segformer-b0-finetuned-cityscapes-512-1024",\n    "b5": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",\n}\n\n# frankfurt is used to fit scores/routers, the rest is test only\nDOMAINS = ["cs_fit", "cs_test", "fog", "night", "rain", "snow"]\n\n_LUT = np.full(256, IGNORE, np.uint8)\nfor l in labels:\n    if 0 <= l.id < 256 and l.trainId not in (-1, 255):\n        _LUT[l.id] = l.trainId\n\n\ndef list_domain(domain):\n    """Returns sorted (image, gt) path pairs for a domain."""\n    if domain.startswith("cs_"):\n        cities = ["frankfurt"] if domain == "cs_fit" else ["lindau", "munster"]\n        imgs = [p for c in cities for p in (DATA / "cityscapes/leftImg8bit/val" / c).glob("*.png")]\n        gt = lambda p: Path(str(p).replace("leftImg8bit", "gtFine", 1)\n                            .replace("_leftImg8bit.png", "_gtFine_labelIds.png"))\n    else:\n        imgs = list((DATA / "acdc/rgb_anon" / domain / "val").rglob("*_rgb_anon.png"))\n        gt = lambda p: Path(str(p).replace("rgb_anon", "gt", 1)\n                            .replace("_rgb_anon.png", "_gt_labelTrainIds.png"))\n    return [(p, gt(p)) for p in sorted(imgs)]\n\n\ndef image_tensor(image):\n    # Tensor bilinear antialiased resize, matching the original torchvision v2 intent.\n    import torch.nn.functional as F\n    a = np.array(image, dtype=np.uint8, copy=True)\n    x = torch.from_numpy(a).permute(2,0,1).float().div(255)\n    x = F.interpolate(x[None], size=SIZE, mode="bilinear", align_corners=False, antialias=True)[0]\n    mean = torch.tensor([.485,.456,.406])[:,None,None]\n    std = torch.tensor([.229,.224,.225])[:,None,None]\n    return (x-mean)/std\n\n\nclass SegSet(Dataset):\n    def __init__(self, domain):\n        self.domain = domain\n        self.items = list_domain(domain)\n        self.img_tf = image_tensor\n\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        img_path, gt_path = self.items[i]\n        gt = np.array(Image.open(gt_path))\n        if self.domain.startswith("cs_"):\n            gt = _LUT[gt]\n        gt = Image.fromarray(gt).resize(SIZE[::-1], Image.NEAREST)\n        return self.img_tf(Image.open(img_path).convert("RGB")), torch.from_numpy(np.array(gt)), img_path.stem\n\n\ndef load_model(name, device="cuda"):\n    model = SegformerForSemanticSegmentation.from_pretrained(MODELS[name])\n    return model.half().to(device).eval()\n\n\ndef confusion(pred, gt):\n    m = gt != IGNORE\n    return np.bincount(NUM_CLASSES * gt[m].astype(np.int64) + pred[m],\n                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)\n\n\ndef miou(conf):\n    iou = np.diag(conf) / (conf.sum(0) + conf.sum(1) - np.diag(conf) + 1e-9)\n    return 100 * iou[conf.sum(1) > 0].mean()\n\n\nTILE = 64  # 512x1024 -> 8x16 tiles\n\n\ndef load(domain, *keys):\n    """Yields cached arrays of every image in a domain, in a fixed order."""\n    for p in sorted((CACHE / domain).glob("*.npz")):\n        with np.load(p) as z:\n            yield [z[k] for k in keys]\n\n\ndef tile_sum(x):\n    h, w = x.shape\n    return x.reshape(h // TILE, TILE, w // TILE, TILE).sum((1, 3))\n\n\ndef gain(gt, pred_b0, pred_b5):\n    """+1 where only B5 is right, -1 where only B0 is right, 0 elsewhere or on ignore."""\n    return ((pred_b5 == gt).astype(np.int8) - (pred_b0 == gt)) * (gt != IGNORE)\n\n\ndef top_tiles(scores, budget):\n    """Boolean mask of the top `budget` share of tiles by score, over the whole domain."""\n    order = np.argsort(-scores.ravel(), kind="stable")\n    m = np.zeros(scores.size, bool)\n    m[order[:round(budget * scores.size)]] = True\n    return m.reshape(scores.shape)\n\n\ndef route_confs(domain, scores, budgets):\n    """Per-image confusions (budgets x images x C x C) when top_tiles get B5."""\n    masks = [top_tiles(scores, b) for b in budgets]\n    confs = np.zeros((len(budgets), len(scores), NUM_CLASSES, NUM_CLASSES), np.int64)\n    for i, (gt, b0, b5) in enumerate(load(domain, "gt", "pred_b0", "pred_b5")):\n        for j, m in enumerate(masks):\n            px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n            confs[j, i] = confusion(np.where(px, b5, b0), gt)\n    return confs\n\n\ndef route_curve(domain, scores, budgets):\n    """mIoU per budget, see route_confs."""\n    return [miou(c.sum(0)) for c in route_confs(domain, scores, budgets)]\n', 'e2_confidence.py': '"""E2: standard confidence scores of B0 as routing signals.\n\nPixel AUROC (logit resolution, sampled pixels) for two targets:\n  err  - B0 is wrong\n  help - B0 is wrong and B5 is right\nTile routing: mean score per tile, % of the B5-B0 gap closed (as in E1).\n"""\nimport json\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_auc_score\n\nfrom common import DOMAINS, IGNORE, RESULTS, TILE, load, route_curve\n\nBUDGETS = [0.05, 0.1, 0.2, 0.3]\nPIXELS_PER_IMAGE = 2000\nSTRIDE = 4  # logits are at 1/4 resolution\n\n\ndef uncertainty(logits):\n    """Per-pixel scores, higher = less confident."""\n    p = logits.softmax(0)\n    top2 = p.topk(2, dim=0).values\n    return {\n        "entropy": -(p * p.clamp_min(1e-12).log()).sum(0),\n        "msp": 1 - top2[0],\n        "margin": top2[1] - top2[0],\n        "energy": -logits.logsumexp(0),\n    }\n\n\ndef main():\n    base = json.loads((RESULTS / "e0.json").read_text())\n    oracle = json.loads((RESULTS / "e1.json").read_text())\n    rng = np.random.default_rng(0)\n    auroc, closed = {}, {}\n\n    for domain in DOMAINS:\n        tiles, pix, err, helps = {}, {}, [], []\n        for gt, b0, b5, logits in load(domain, "gt", "pred_b0", "pred_b5", "logits_b0"):\n            scores = uncertainty(torch.from_numpy(logits).float())\n            for k, s in scores.items():\n                tiles.setdefault(k, []).append(F.avg_pool2d(s[None], TILE // STRIDE)[0].numpy())\n\n            gt, b0, b5 = (a[STRIDE // 2::STRIDE, STRIDE // 2::STRIDE] for a in (gt, b0, b5))\n            valid = np.flatnonzero(gt != IGNORE)\n            idx = rng.choice(valid, min(PIXELS_PER_IMAGE, valid.size), replace=False)\n            for k, s in scores.items():\n                pix.setdefault(k, []).append(s.numpy().ravel()[idx])\n            wrong = b0.ravel()[idx] != gt.ravel()[idx]\n            err.append(wrong)\n            helps.append(wrong & (b5.ravel()[idx] == gt.ravel()[idx]))\n\n        err, helps = np.concatenate(err), np.concatenate(helps)\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        auroc[domain], closed[domain] = {}, {}\n        for k in tiles:\n            s = np.concatenate(pix[k])\n            auroc[domain][k] = [round(roc_auc_score(err, s), 3), round(roc_auc_score(helps, s), 3)]\n            curve = route_curve(domain, np.stack(tiles[k]), BUDGETS)\n            closed[domain][k] = [round(100 * (m - b0_miou) / (b5_miou - b0_miou), 1) for m in curve]\n\n    print(f"{\'domain\':8} {\'method\':8} {\'AUC err\':>8} {\'AUC help\':>9}" + "".join(f"{b:>7.0%}" for b in BUDGETS))\n    for domain in DOMAINS:\n        for k, (a_err, a_help) in auroc[domain].items():\n            print(f"{domain:8} {k:8} {a_err:8.3f} {a_help:9.3f}" + "".join(f"{c:7.1f}" for c in closed[domain][k]))\n        print(f"{domain:8} {\'oracle\':8} {\'\':8} {\'\':9}" + "".join(f"{c:7.1f}" for c in oracle[domain]["oracle_tile"][:len(BUDGETS)]))\n\n    (RESULTS / "e2.json").write_text(json.dumps({"budgets": BUDGETS, "auroc": auroc, "closed": closed}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'e8_cityscapes.py': '"""E8: gate for the adapter idea - is a frozen stage-3 context good enough for B5?\n\nB5 sees the whole frame, but in the chosen stages only the tokens of routed tiles are updated\nby the transformer blocks. Every other token keeps its stage-entry value for all blocks, so it\nonly serves as keys/values ("frozen context"). Here the entry values come from the real B5\n(an upper bound for any adapter that maps B0 tokens into B5 token space).\n\nQuality only: the whole frame is still computed, only the write-back is masked. The saving is\ncounted analytically for now (blocks run on routed queries only; context costs K/V projections),\nreal FLOPs and latency are measured later.\n\nStages (0-based): 2 = stride 16, 40 blocks, most of B5\'s encoder compute; 3 = stride 32.\nVariants: "s3" freezes stage 2 only, "s3+s4" freezes stages 2 and 3. Stages 0-1 are run on the\nwhole frame in this experiment (known simplification, about a fifth of the encoder cost).\n\nTiles are selected as in E5 (domain-wide ranking at 10% and 20%): `rank` (entropy + knn) and the oracle.\nReported as % of the B5-B0 gap closed, next to:\n  full  - B5 on the full frame inside the routed tiles (the E5 "full" column, the target)\n  agree - pixel agreement with that full-frame B5 prediction inside the routed tiles\nGate: s3 should stay close to `full` on ACDC. For reference E5 ctx64 (cost x0.9) closed 42 / 17 / 35 / 60\n(fog / night / rain / snow) against 61 / 30 / 59 / 59 for full.\n"""\nimport json\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom common import (CACHE, DOMAINS, NUM_CLASSES, RESULTS, SIZE, TILE, SegSet, confusion, gain, load,\n                    load_model, miou, tile_sum, top_tiles)\ndef rank(scores):\n    return scores.ravel().argsort().argsort().reshape(scores.shape)\n\nDOMAINS = ["cs_fit", "cs_test"]\n\nBUDGETS = [0.1, 0.2]\nSTAGES = {"s3": [2], "s3+s4": [2, 3]}\n\n\nclass Frozen(nn.Module):\n    """Wraps a SegformerLayer: tokens outside state.mask are reset to the stage entry value."""\n\n    def __init__(self, layer, state, first):\n        super().__init__()\n        self.layer, self.state, self.first = layer, state, first\n\n    def forward(self, hidden_states, *args, **kwargs):\n        s = self.state\n        if self.first:\n            s.entry = hidden_states\n        out = self.layer(hidden_states, *args, **kwargs)\n        if s.mask is None:\n            return out\n        return (torch.where(s.mask, out[0], s.entry),) + tuple(out[1:])\n\n\ndef install(model, stages=(2, 3)):\n    """Wrap every block of the given stages, returns {stage: state}."""\n    state = {st: SimpleNamespace(mask=None, entry=None) for st in stages}\n    blocks = model.segformer.encoder.block\n    for st in stages:\n        for i, layer in enumerate(blocks[st]):\n            blocks[st][i] = Frozen(layer, state[st], i == 0)\n    return state\n\n\ndef token_mask(m, stage):\n    """(8, 16) bool tile mask -> (1, N, 1) bool mask over the tokens of a stage (stride 4 * 2**stage)."""\n    k = TILE // (4 * 2 ** stage)\n    t = torch.from_numpy(m).cuda().repeat_interleave(k, 0).repeat_interleave(k, 1)\n    return t.reshape(1, -1, 1)\n\n\n@torch.inference_mode()\ndef predict(model, state, img, masks):\n    """B5 prediction on the full frame where only `masks` {stage: token mask} tokens are updated."""\n    for st, s in state.items():\n        s.mask = masks.get(st)\n    try:\n        logits = model(pixel_values=img[None].cuda().half()).logits\n        return F.interpolate(logits.float(), SIZE, mode="bilinear", align_corners=False).argmax(1)[0].byte().cpu().numpy()\n    finally:\n        for s in state.values():\n            s.mask = None\n            s.entry = None\n\n\n\ndef sanity(model, state):\n    """An all-true mask must reproduce the cached full-frame B5 prediction."""\n    img = SegSet("cs_fit")[0][0]\n    (cached,) = next(load("cs_fit", "pred_b5"))\n    full = {st: token_mask(np.ones((SIZE[0] // TILE, SIZE[1] // TILE), bool), st) for st in state}\n    agree = (predict(model, state, img, full) == cached).mean()\n    assert agree > 0.999, f"all-true mask does not reproduce B5, agreement {agree:.4f}"\n\n\ndef main():\n    model = load_model("b5")\n    state = install(model)\n    sanity(model, state)\n\n    base = json.loads((RESULTS / "e0_cityscapes.json").read_text())\n    feats = np.load(CACHE / "e8_tiles_cityscapes.npz")\n    report_path = RESULTS / "e8_cityscapes.json"\n    report = json.loads(report_path.read_text()) if report_path.exists() else {}\n\n    print(f"{\'domain\':8} {\'method\':8} {\'budget\':>6} {\'full\':>7}"\n          + "".join(f"{n:>9}" for n in STAGES) + "".join(f"{f\'agree {n}\':>14}" for n in STAGES))\n    for domain in DOMAINS:\n        if domain in report:\n            print(domain, "уже завершён, пропускаем", flush=True)\n            continue\n        dataset = SegSet(domain)\n        assert [p.stem for p, _ in dataset.items] == [p.stem for p in sorted((CACHE / domain).glob("*.npz"))]\n\n        entropy, knn = feats[f"{domain}__entropy__mean"], feats[f"{domain}__knn__mean"]\n        gains = np.stack([tile_sum(gain(*a)) for a in load(domain, "gt", "pred_b0", "pred_b5")])\n        scores = {"rank": rank(entropy) + rank(knn), "oracle": gains}\n        masks = {(k, b): top_tiles(s, b) for k, s in scores.items() for b in BUDGETS}\n\n        zeros = lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)\n        full = {key: zeros() for key in masks}\n        conf = {(*key, n): zeros() for key in masks for n in STAGES}\n        agree = {(*key, n): np.zeros(2, np.int64) for key in masks for n in STAGES}\n\n        started = __import__("time").monotonic()\n        loader = DataLoader(dataset, batch_size=None, num_workers=2)\n        for i, ((img, _, _), (gt, b0, b5)) in enumerate(zip(loader, load(domain, "gt", "pred_b0", "pred_b5"))):\n            for key, m in masks.items():\n                px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n                full[key] += confusion(np.where(px, b5, b0), gt)\n                for n, stages in STAGES.items():\n                    pred = predict(model, state, img, {st: token_mask(m[i], st) for st in stages})\n                    conf[(*key, n)] += confusion(np.where(px, pred, b0), gt)\n                    agree[(*key, n)] += [(pred == b5)[px].sum(), px.sum()]\n\n            if (i+1) % 5 == 0 or i+1 == len(dataset):\n                elapsed = __import__("time").monotonic()-started\n                eta = elapsed/(i+1)*(len(dataset)-i-1)/60\n                print(f"E8 {domain}: {i+1}/{len(dataset)}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин", flush=True)\n\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        closed = lambda c: round(100 * (miou(c) - b0_miou) / (b5_miou - b0_miou), 1)\n        report[domain] = {}\n        for key in masks:\n            row = {"full": closed(full[key])}\n            row.update({n: closed(conf[(*key, n)]) for n in STAGES})\n            row.update({f"agree_{n}": round(100 * agree[(*key, n)][0] / agree[(*key, n)][1], 1) for n in STAGES})\n            report[domain][f"{key[0]}@{key[1]}"] = row\n            print(f"{domain:8} {key[0]:8} {key[1]:6.0%} {row[\'full\']:7.1f}"\n                  + "".join(f"{row[n]:9.1f}" for n in STAGES)\n                  + "".join(f"{row[f\'agree_{n}\']:14.1f}" for n in STAGES))\n\n        temp = report_path.with_suffix(".json.part")\n        temp.write_text(json.dumps(report, indent=2))\n        temp.replace(report_path)\n        print("Сохранён домен:", domain, flush=True)\n\n    print("Результат:", report_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'runner.py': '"""Cityscapes-only preparation and E8; frozen B0/B5. No ACDC tuning."""\nimport os\nos.environ[\'USE_TF\'] = \'0\'\nimport argparse, gc, hashlib, json, platform, tarfile, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load, load_model, confusion, miou\n\nDOMAINS = [\'cs_fit\', \'cs_test\']\nSIGNATURE = \'e8-cs-colab-v1\'\n\ndef write_json(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.part\')\n    tmp.write_text(json.dumps(data, indent=2))\n    tmp.replace(path)\n\ndef checkpoint(domain):\n    backup = os.environ.get(\'E8_BACKUP\')\n    if not backup:\n        return\n    dest = Path(backup); dest.mkdir(parents=True, exist_ok=True)\n    name = dest / f\'cache_{domain}.tar\'\n    tmp = name.with_suffix(\'.tar.part\')\n    with tarfile.open(tmp, \'w\') as tar:\n        tar.add(CACHE / domain, arcname=domain)\n    tmp.replace(name)\n    print(\'Кэш сохранён на Drive:\', name, flush=True)\n\ndef valid_cache(path):\n    if not path.exists():\n        return False\n    with np.load(path, allow_pickle=False) as z:\n        assert set([\'gt\', \'pred_b0\', \'pred_b5\', \'logits_b0\', \'feat_b0\']) <= set(z.files)\n        assert all(z[k].shape == SIZE for k in [\'gt\', \'pred_b0\', \'pred_b5\'])\n        assert z[\'logits_b0\'].shape == (19, 128, 256)\n        assert z[\'feat_b0\'].shape == (256, 64, 128)\n    return True\n\n@torch.inference_mode()\ndef cache():\n    models = [load_model(n) for n in [\'b0\', \'b5\']]\n    captured = {}\n    handle = models[0].decode_head.classifier.register_forward_pre_hook(lambda m, x: captured.update(f=x[0]))\n    report = {}\n    try:\n        for domain in DOMAINS:\n            data = SegSet(domain)\n            assert len(data) == (267 if domain == \'cs_fit\' else 233)\n            folder = CACHE / domain; folder.mkdir(parents=True, exist_ok=True)\n            confs = [np.zeros((19,19), np.int64) for _ in range(2)]\n            start = time.monotonic()\n            for i in range(len(data)):\n                img, gt, name = data[i]\n                path = folder / f\'{name}.npz\'\n                if not valid_cache(path):\n                    x = img[None].cuda().half()\n                    l0 = models[0](pixel_values=x).logits\n                    feat = F.avg_pool2d(captured[\'f\'], 2)[0].cpu().numpy()\n                    preds = []\n                    for logits in [l0, models[1](pixel_values=x).logits]:\n                        preds.append(F.interpolate(logits.float(), SIZE, mode=\'bilinear\', align_corners=False).argmax(1)[0].byte().cpu().numpy())\n                    tmp = path.with_suffix(\'.npz.part\')\n                    with tmp.open(\'wb\') as f:\n                        np.savez_compressed(f, gt=gt.numpy(), pred_b0=preds[0], pred_b5=preds[1], logits_b0=l0[0].cpu().numpy(), feat_b0=feat)\n                    tmp.replace(path)\n                with np.load(path) as z:\n                    for j, k in enumerate([\'pred_b0\', \'pred_b5\']):\n                        confs[j] += confusion(z[k], z[\'gt\'])\n                if (i+1) % 10 == 0 or i+1 == len(data):\n                    print(f\'E0 {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\', flush=True)\n            report[domain] = dict(zip([\'b0\',\'b5\'], [float(miou(c)) for c in confs]))\n            write_json(RESULTS / \'e0_cityscapes.json\', report)\n            checkpoint(domain)\n    finally:\n        handle.remove()\n    print(report, flush=True)\n\n@torch.inference_mode()\ndef scores():\n    """Only entropy + kNN required by E8. Bank fitted solely on cs_fit."""\n    from e2_confidence import uncertainty\n    from common import IGNORE\n    reference = CACHE / \'e8_knn_bank.npy\'\n    if not reference.exists():\n        rng = np.random.default_rng(0)\n        samples = []\n        for gt, feat in load(\'cs_fit\', \'gt\', \'feat_b0\'):\n            y = gt[4::8,4::8].ravel()\n            valid = np.flatnonzero(y != IGNORE)\n            idx = rng.choice(valid, min(500, len(valid)), replace=False)\n            samples.append(feat.reshape(256,-1).T[idx].astype(np.float32))\n        x = np.concatenate(samples)\n        # Separate seeded RNG: exact bank saved, not claimed identical to old E3.\n        idx = np.random.default_rng(0).permutation(len(x))[:20000]\n        bank = F.normalize(torch.from_numpy(x[idx]), dim=1).numpy()\n        np.save(reference, bank)\n    bank = torch.from_numpy(np.load(reference)).cuda().float()\n    assert bank.shape == (20000,256)\n    output = {}\n    for domain in DOMAINS:\n        arrays = {\'entropy\':[], \'knn\':[]}\n        count = len(SegSet(domain))\n        for i, (logits, feat) in enumerate(load(domain, \'logits_b0\',\'feat_b0\')):\n            entropy = uncertainty(torch.from_numpy(logits).float())[\'entropy\']\n            entropy = F.avg_pool2d(entropy[None],2)[0].numpy()\n            f = F.normalize(torch.from_numpy(feat.astype(np.float32)).cuda().flatten(1).T, dim=1)\n            distances = []\n            for chunk in f.split(512):\n                distances.append((1-(chunk @ bank.T).topk(10,dim=1).values.mean(1)).cpu())\n            knn = torch.cat(distances).numpy().reshape(64,128)\n            for name, m in [(\'entropy\',entropy),(\'knn\',knn)]:\n                # 8x8 feature cells per 64x64 tile.\n                arrays[name].append(m.reshape(8,8,16,8).mean((1,3)))\n            if (i+1)%10==0 or i+1==count:\n                print(f\'kNN {domain}: {i+1}/{count}\', flush=True)\n        for k,v in arrays.items():\n            output[f\'{domain}__{k}__mean\'] = np.stack(v)\n    path = CACHE / \'e8_tiles_cityscapes.npz\'\n    np.savez(path, **output)\n    write_json(RESULTS/\'knn_config.json\', {\'fit\':\'cs_fit only\', \'bank\':20000, \'k\':10, \'sampling_seed\':0,\n        \'bank_sha256\':hashlib.sha256(reference.read_bytes()).hexdigest(),\n        \'note\':\'Minimal E3 alternative; saved bank differs from historical E3 RNG. Do not compare as exact reproduction.\'})\n    print(\'Сохранено:\', path, flush=True)\n\n@torch.inference_mode()\ndef smoke():\n    from e8_cityscapes import install, predict, token_mask\n    model = load_model(\'b5\'); state = install(model)\n    data = SegSet(\'cs_fit\'); img, _, name = data[0]\n    all_masks = {st: token_mask(np.ones((8,16),bool),st) for st in state}\n    plain = predict(model,state,img,{})\n    masked = predict(model,state,img,all_masks)\n    assert np.array_equal(plain,masked), \'All-true differs from unmasked model\'\n    path = CACHE/\'cs_fit\'/f\'{name}.npz\'\n    assert path.is_file(), \'Сначала выполните E0\'\n    with np.load(path) as z:\n        agreement = float((masked==z[\'pred_b5\']).mean())\n    assert agreement > .999, f\'Cached agreement={agreement}\'\n    selected = np.zeros((8,16),bool); selected[3:5,6:10] = True\n    routed = predict(model,state,img,{2:token_mask(selected,2)})\n    assert routed.shape == SIZE\n    write_json(RESULTS/\'smoke.json\', {\'complete\':True,\'all_true_equal\':True,\'cached_agreement\':agreement,\'selected_mask_forward_ok\':True})\n    print(\'Smoke OK. Совпадение с кэшем:\',agreement, flush=True)\n\n\ndef selftest():\n    from transformers import SegformerConfig, SegformerForSemanticSegmentation\n    from e8_cityscapes import install\n    torch.manual_seed(0)\n    cfg=SegformerConfig(num_channels=3,num_encoder_blocks=4,depths=[1,1,2,1],hidden_sizes=[8,16,32,64],num_attention_heads=[1,2,4,8],sr_ratios=[8,4,2,1],decoder_hidden_size=16,num_labels=19,drop_path_rate=0.)\n    m=SegformerForSemanticSegmentation(cfg).eval()\n    x=torch.randn(1,3,64,128)\n    with torch.inference_mode():\n        original=m(pixel_values=x).logits\n        states=install(m)\n        for st,s in states.items():\n            s.mask=torch.ones(1,(64//(4*2**st))*(128//(4*2**st)),1,dtype=torch.bool)\n        assert torch.equal(original,m(pixel_values=x).logits)\n        outputs=[]; handles=[]\n        for st,s in states.items():\n            s.mask=torch.zeros_like(s.mask)\n            for block in m.segformer.encoder.block[st]:\n                handles.append(block.register_forward_hook(lambda mod,inp,out:outputs.append(torch.equal(out[0],mod.state.entry))))\n        m(pixel_values=x)\n        for h in handles: h.remove()\n        assert outputs and all(outputs)\n    print(\'Tiny SegFormer: all-true / all-false OK\',flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'cache\',\'scores\',\'smoke\',\'e8\',\'selftest\']);args=p.parse_args()\n    if args.stage!=\'selftest\': assert torch.cuda.is_available(), \'Нужен GPU runtime\'\n    RESULTS.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n    import transformers\n    write_json(RESULTS/\'environment.json\', {\'signature\':SIGNATURE,\'python\':platform.python_version(),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\'size\':SIZE,\'tile\':TILE,\'domains\':DOMAINS,\'training\':False})\n    if args.stage==\'e8\':\n        from e8_cityscapes import main\n        main()\n    else: globals()[args.stage]()\n'}
for name, text in SOURCES.items():
    path = EXP/name
    if path.exists():
        assert path.read_text() == text, f'Изменён код: {path}. Для новой версии поменяй RUN_NAME.'
    else: path.write_text(text)
code_copy = SAVE/'source'
code_copy.mkdir(exist_ok=True)
for name in SOURCES: shutil.copy2(EXP/name, code_copy/name)
manifest = {'sha256':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
            'domains':['cs_fit','cs_test'], 'size':[512,1024], 'tile':64,
            'budgets':[0.1,0.2], 'frozen_models':True, 'rank_transductive':True}
manifest_path = SAVE/'run_config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Конфигурация изменилась: выбери новый RUN_NAME'
else: manifest_path.write_text(json.dumps(manifest, indent=2))
print('Код записан:', EXP)


In [ ]:
# 5. Восстановление после сброса VM и запускатель с логами.
import tarfile, time
for domain in ['cs_fit','cs_test']:
    backup = SAVE/'backups'/f'cache_{domain}.tar'
    if backup.is_file() and not (CACHE/domain).exists():
        print('Восстанавливаем:', backup, flush=True)
        with tarfile.open(backup) as tar:
            members = tar.getmembers()
            for m in members:
                assert m.isfile() or m.isdir()
                assert (CACHE/m.name).resolve().is_relative_to(CACHE.resolve())
            tar.extractall(CACHE)
for name in ['e8_knn_bank.npy','e8_tiles_cityscapes.npz']:
    src = SAVE/name
    if src.is_file() and not (CACHE/name).exists(): shutil.copy2(src,CACHE/name)

def run(stage):
    log_path = SAVE/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:', stage, '| Лог:', log_path, flush=True)
    started = time.monotonic()
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','runner.py',stage],cwd=EXP,
                                   env=os.environ.copy(), stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    tail = log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-started)/60:.1f} мин | '+ ' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait()
            raise
    print(log_path.read_text(errors='replace')[-8000:])
    assert status == 0, f'Ошибка {status}: смотри лог {log_path}'
    print(f'Завершено за {(time.monotonic()-started)/60:.1f} мин')


In [ ]:
# 6. Проверяем готовый кэш и эталон прошлого запуска.
for domain,n in [('cs_fit',267),('cs_test',233)]:
    assert len(list((CACHE/domain).glob('*.npz')))==n, f'Неполный E0-кэш {domain}; восстанови backups из E8.'
assert (CACHE/'e8_tiles_cityscapes.npz').is_file(), 'Нет готовых entropy+kNN.'
assert (SAVE/'e9_adapter_v1/e9_routing.json').is_file() or (SAVE/'e8b_v1/e8b_cityscapes.json').is_file(), 'Нужен эталон E9/E8b JSON на Drive.'
OUT=SAVE/'e8c_k_sweep_v1'
OUT.mkdir(exist_ok=True)
os.environ['E8C_OUT']=str(OUT)
os.environ['E8C_MEASURED_REUSE']=str(SAVE/'e8c_cost_v1')
print('Новые результаты:',OUT)
print('Папка прежних измерений:',os.environ['E8C_MEASURED_REUSE'])


In [ ]:
# 7. Записываем новые скрипты; весь код встроен в notebook.
NEW_SOURCES={'e8c_k_sweep.py': '"""E8c: piecewise frozen context, teacher states only. No training/speed claims."""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport argparse,csv,hashlib,json,time\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom common import CACHE,RESULTS,SIZE,TILE,SegSet,load_model,confusion,miou,top_tiles\nOUT=Path(os.environ[\'E8C_OUT\'])\nCONFIG={\'schema\':\'e8c-midpoint-k-sweep-v1\',\'domains\':[\'cs_test\'],\'methods\':[\'rank\'],\'budgets\':[.1,.2],\n        \'variants\':[\'k1\',\'k2\',\'k4\',\'k8\'], \'training\':False,\n        \'restore\':\'true B5 final outside selected tiles\',\'early_stages\':\'full B5\',\n        \'ks\':[1,2,4,8],\'checkpoint_every_frames\':10,\'bootstrap_repeats\':1000,\'reference_states\':\'true B5 per image\'}\n\ndef write_json(path,data):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    p=path.with_suffix(path.suffix+\'.part\');p.write_text(json.dumps(data,indent=2));p.replace(path)\n\ndef sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef initialize():\n    OUT.mkdir(parents=True,exist_ok=True)\n    cfg=dict(CONFIG,source_sha256=sha(__file__),common_sha256=sha(Path(__file__).parent/\'common.py\'),\n             tile_features_sha256=sha(CACHE/\'e8_tiles_cityscapes.npz\'))\n    p=OUT/\'quality_config.json\'\n    if p.exists():assert json.loads(p.read_text())==cfg,\'Use a new E8C_OUT for changed configuration\'\n    else:write_json(p,cfg)\n\n\ndef sources(k,n):\n    assert 1<=k<=n\n    return [int((i*k//n+.5)*n/k) for i in range(n)]\n\ndef context_index(i,n,mode):\n    if mode==\'per_block\':return i\n    if mode==\'mid\':return n//2\n    return sources(int(mode[1:]),n)[i]\n\n\nclass Frozen(nn.Module):\n    def __init__(self,layer,state,index,n):\n        super().__init__();self.layer=layer;self.state=state;self.index=index;self.n=n\n    def forward(self,x,*args,**kwargs):\n        s=self.state\n        if s.recording:\n            s.inputs[self.index]=x\n            out=self.layer(x,*args,**kwargs)\n            if self.index==self.n-1:s.final=out[0]\n            return out\n        if s.mask is None:return self.layer(x,*args,**kwargs)\n        ctx=s.inputs[context_index(self.index,self.n,s.mode)]\n        out=self.layer(torch.where(s.mask,x,ctx),*args,**kwargs)\n        y=torch.where(s.mask,out[0],ctx)\n        if self.index==self.n-1:y=torch.where(s.mask,y,s.final)\n        return (y,)+tuple(out[1:])\n\ndef install(model):\n    blocks=model.segformer.encoder.block[2]\n    state=SimpleNamespace(recording=False,mask=None,mode=\'mid\',inputs=[None]*len(blocks),final=None)\n    for i,block in enumerate(blocks):blocks[i]=Frozen(block,state,i,len(blocks))\n    return state\n\n@torch.inference_mode()\ndef logits(model,state,img,recording=False,mask=None,mode=\'mid\'):\n    state.recording=recording;state.mask=mask;state.mode=mode\n    try:return model(pixel_values=img).logits\n    finally:state.recording=False;state.mask=None\n\ndef prediction(z):\n    return F.interpolate(z.float(),SIZE,mode=\'bilinear\',align_corners=False).argmax(1)[0].byte().cpu().numpy()\n\ndef mask_tokens(m,device):\n    k=TILE//16\n    return torch.as_tensor(m,device=device).repeat_interleave(k,0).repeat_interleave(k,1).reshape(1,-1,1)\n\ndef rank(x):return x.ravel().argsort().argsort().reshape(x.shape)\n\ndef reference_values(domain):\n    """Require the actual prior report, rather than trusting hardcoded rounded numbers."""\n    p=RESULTS.parent/\'e9_adapter_v1/e9_routing.json\'\n    if p.exists():\n        data=json.loads(p.read_text());rows=data.get(domain,[])\n        refs={float(r[\'budget\']):float(r[\'miou\']) for r in rows if r[\'variant\']==\'true_context_true_output\' and r[\'method\']==\'rank\'}\n        if all(b in refs for b in CONFIG[\'budgets\']):return refs,str(p)\n    p=RESULTS.parent/\'e8b_v1/e8b_cityscapes.json\'\n    if p.exists():\n        data=json.loads(p.read_text()).get(domain,{})\n        if all(f\'rank@{b}\' in data for b in CONFIG[\'budgets\']):\n            return {b:float(data[f\'rank@{b}\'][\'miou_mid\']) for b in CONFIG[\'budgets\']},str(p)\n    raise RuntimeError(\'Не найден эталон E9/E8b для k=1. Восстанови e9_routing.json или e8b_cityscapes.json на Drive.\')\n\n@torch.inference_mode()\ndef strict_sanity(model,state,img,plain_logits,cached):\n    recorded=logits(model,state,img,recording=True)\n    assert torch.equal(recorded,plain_logits),\'Recording changed the ordinary pass\'\n    baseline=prediction(plain_logits)\n    assert (baseline==cached).mean()>.999,\'First-frame cache mismatch\'\n    ones=mask_tokens(np.ones((8,16),bool),img.device)\n    for mode in CONFIG[\'variants\']:\n        test=logits(model,state,img,mask=ones,mode=mode)\n        assert torch.equal(test,plain_logits),\'All-true mismatch: \'+mode\n    checker=(np.indices((8,16)).sum(axis=0)%2)==0  # exactly half the tiles\n    test=logits(model,state,img,mask=mask_tokens(checker,img.device),mode=\'per_block\')\n    error=float((test.float()-plain_logits.float()).abs().max().item())\n    assert torch.allclose(test,plain_logits,rtol=1e-4,atol=1e-4),f\'k=40 replay logits mismatch: {error}\'\n    assert np.array_equal(prediction(test),baseline),\'k=40 replay must reproduce the WHOLE prediction\'\n    state.inputs=[None]*len(state.inputs);state.final=None\n    return {\'recording_equal\':True,\'all_true_equal_for_all_k\':True,\'k40_half_tiles_whole_prediction_equal\':True,\n            \'k40_max_abs_logit_error\':error,\'cached_agreement\':float((baseline==cached).mean())}\n\n\ndef save_progress(path,completed,confs,agree,names):\n    temp=path.with_suffix(\'.npz.part\')\n    with temp.open(\'wb\') as f:\n        np.savez_compressed(f,completed=np.array(completed),confs=confs[:completed],agreements=agree,\n                            names=np.array(names),modes=np.array([\'b0\',\'full_tiles\']+CONFIG[\'variants\']),\n                            budgets=np.array(CONFIG[\'budgets\']))\n    temp.replace(path)\n\n@torch.inference_mode()\ndef quality(device=\'cuda\'):\n    report_path=OUT/\'e8c_quality.json\'\n    report=json.loads(report_path.read_text()) if report_path.exists() else {}\n    feats=np.load(CACHE/\'e8_tiles_cityscapes.npz\')\n    dtype=torch.float16 if str(device).startswith(\'cuda\') else torch.float32\n    model=load_model(\'b5\',device=device).requires_grad_(False)\n    first_domain=CONFIG[\'domains\'][0];first=SegSet(first_domain)\n    assert len(first)>0,\'No images\'\n    first_img=first[0][0][None].to(device=device,dtype=dtype)\n    plain=model(pixel_values=first_img).logits\n    state=install(model);nblocks=len(state.inputs)\n    assert nblocks==40,f\'Expected B5 depth40, got {nblocks}\'\n    with np.load(CACHE/first_domain/(first.items[0][0].stem+\'.npz\')) as z:cached=z[\'pred_b5\']\n    sanity=strict_sanity(model,state,first_img,plain,cached)\n    write_json(OUT/\'sanity.json\',sanity)\n    del first_img,plain\n    for domain in CONFIG[\'domains\']:\n        if domain in report:\n            print(domain,\'готов, пропускаем\',flush=True);continue\n        reference,reference_source=reference_values(domain)\n        dataset=SegSet(domain);names=[p.stem for p,_ in dataset.items]\n        assert names==[p.stem for p in sorted((CACHE/domain).glob(\'*.npz\'))]\n        scores=rank(feats[f\'{domain}__entropy__mean\'])+rank(feats[f\'{domain}__knn__mean\'])\n        masks=[top_tiles(scores,b) for b in CONFIG[\'budgets\']]\n        modes=CONFIG[\'variants\'];c=2+len(modes);n=len(dataset)\n        confs=np.zeros((n,len(masks),c,19,19),np.int64)\n        agree=np.zeros((len(masks),len(modes)+1,2),np.int64)\n        progress_path=OUT/f\'{domain}_progress.npz\';done=0\n        if progress_path.exists():\n            with np.load(progress_path,allow_pickle=False) as z:\n                assert list(z[\'names\'])==names,\'Progress image order differs\'\n                assert list(z[\'modes\'])==[\'b0\',\'full_tiles\']+modes\n                assert np.array_equal(z[\'budgets\'],CONFIG[\'budgets\'])\n                done=int(z[\'completed\']);assert 0<=done<=n\n                assert z[\'confs\'].shape==confs[:done].shape\n                confs[:done]=z[\'confs\'];agree[:]=z[\'agreements\']\n            print(f\'Продолжаем {domain} после {done}/{n} кадров\',flush=True)\n        start=time.monotonic()\n        for i in range(done,n):\n            name=names[i];img,_,_=dataset[i];img=img[None].to(device=device,dtype=dtype)\n            full=logits(model,state,img,recording=True);pred_full=prediction(full)\n            with np.load(CACHE/domain/(name+\'.npz\')) as z:gt=z[\'gt\'];b0=z[\'pred_b0\'];b5=z[\'pred_b5\']\n            assert (pred_full==b5).mean()>.999,f\'Cache mismatch: {name}\'\n            for j,m in enumerate(masks):\n                px=m[i].repeat(TILE,0).repeat(TILE,1);tm=mask_tokens(m[i],device)\n                confs[i,j,0]=confusion(b0,gt);confs[i,j,1]=confusion(np.where(px,b5,b0),gt)\n                agree[j,0]+=[int((b0==b5)[px].sum()),int(px.sum())]\n                for k,mode in enumerate(modes):\n                    modified=prediction(logits(model,state,img,mask=tm,mode=mode))\n                    confs[i,j,k+2]=confusion(np.where(px,modified,b0),gt)\n                    agree[j,k+1]+=[int((modified==b5)[px].sum()),int(px.sum())]\n            state.inputs=[None]*nblocks;state.final=None\n            if (i+1)%CONFIG[\'checkpoint_every_frames\']==0 or i+1==n:\n                save_progress(progress_path,i+1,confs,agree,names)\n            if (i+1)%5==0 or i+1==n:\n                elapsed=time.monotonic()-start\n                eta=elapsed/(i+1-done)*(n-i-1)/60\n                print(f\'E8c-k {domain}: {i+1}/{n}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин\',flush=True)\n        # Fail before publishing any interpretation if the fixed k=1 reference changed.\n        checks=[]\n        for j,b in enumerate(CONFIG[\'budgets\']):\n            actual=float(miou(confs[:,j,2].sum(0)))\n            delta=actual-reference[b]\n            checks.append({\'budget\':b,\'actual_k1_miou\':actual,\'reference_miou\':reference[b],\n                           \'delta_pp\':delta,\'ok\':abs(delta)<.01})\n        write_json(OUT/f\'{domain}_reference_check.json\',{\'source\':reference_source,\'checks\':checks})\n        assert all(r[\'ok\'] for r in checks),\'MISMATCH: k=1 differs from actual E9/E8b report; stop and inspect cache/model/masks\'\n        rows=[]\n        for j,b in enumerate(CONFIG[\'budgets\']):\n            values=[float(miou(z)) for z in confs[:,j].sum(0)]\n            for k,mode in enumerate(modes):\n                a=agree[j,k+1];full_gain=values[1]-values[0]\n                row={\'budget\':b,\'method\':\'rank\',\'variant\':mode,\'k\':int(mode[1:]),\'miou\':values[k+2],\n                     \'gain_vs_b0_pp\':values[k+2]-values[0],\'miou_b0\':values[0],\'miou_full_tiles\':values[1],\n                     \'delta_vs_full_tiles_pp\':values[k+2]-values[1],\'delta_vs_k1_pp\':values[k+2]-values[2],\n                     \'recovered_full_gain_pct\':100*(values[k+2]-values[0])/full_gain if abs(full_gain)>1e-12 else None,\n                     \'agree_b5_percent\':100*int(a[0])/max(int(a[1]),1),\n                     \'state_inputs\':sorted(set(sources(int(mode[1:]),nblocks)))}\n                rows.append(row);print(row,flush=True)\n        # Complete checkpoint already contains all raw frame matrices.\n        report[domain]={\'rows\':rows,\'frames\':n,\'reference_check\':checks,\'sanity\':sanity,\n                        \'schedules\':{mode:sources(int(mode[1:]),nblocks) for mode in modes}}\n        write_json(report_path,report)\n    rows=[dict(domain=d,**r) for d,v in report.items() for r in v[\'rows\']]\n    fields=list(dict.fromkeys(k for r in rows for k in r))\n    with (OUT/\'e8c_quality.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)\n\n\ndef bootstrap():\n    # Domain completion/reference checks must have passed before bootstrap is allowed.\n    report=json.loads((OUT/\'e8c_quality.json\').read_text())\n    rows=[];rng=np.random.default_rng(0)\n    for domain in CONFIG[\'domains\']:\n        assert domain in report\n        with np.load(OUT/f\'{domain}_progress.npz\') as z:\n            conf=z[\'confs\'];names=z[\'names\'];budgets=z[\'budgets\'];modes=list(z[\'modes\'])\n            assert int(z[\'completed\'])==len(names)==len(conf)\n        for unit in [\'frame\',\'sequence\']:\n            groups=np.arange(len(names)).astype(str) if unit==\'frame\' else np.array([\'_\'.join(n.split(\'_\')[:2]) for n in names])\n            unique=sorted(set(groups));g=np.stack([conf[groups==k].sum(0) for k in unique])\n            for j,b in enumerate(budgets):\n                for mode in CONFIG[\'variants\']:\n                    refs=[\'full_tiles\'] if mode==\'k1\' else [\'k1\',\'full_tiles\']\n                    a=modes.index(mode)\n                    for ref in refs:\n                        k=modes.index(ref);diff=[]\n                        for _ in range(CONFIG[\'bootstrap_repeats\']):\n                            idx=rng.integers(0,len(unique),len(unique));c=g[idx,j].sum(0)\n                            diff.append(miou(c[a])-miou(c[k]))\n                        total=g[:,j].sum(0);ci=np.percentile(diff,[2.5,97.5])\n                        rows.append({\'domain\':domain,\'budget\':float(b),\'unit\':unit,\'groups\':len(unique),\n                                     \'comparison\':mode+\' minus \'+ref,\'delta_miou_pp\':float(miou(total[a])-miou(total[k])),\n                                     \'ci_low\':float(ci[0]),\'ci_high\':float(ci[1]),\'repeats\':CONFIG[\'bootstrap_repeats\']})\n    write_json(OUT/\'e8c_bootstrap.json\',rows)\n    for row in rows:print(row,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'quality\',\'bootstrap\']);a=p.parse_args()\n    initialize()\n    if a.stage==\'quality\':assert torch.cuda.is_available(),\'Select GPU runtime\'\n    globals()[a.stage]()\n', 'e10_cost_k.py': '"""Measured dense cost and explicit arithmetic scenarios; no claimed sparse latency."""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport csv,gc,hashlib,json,time,platform\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom common import CACHE,RESULTS,SIZE,TILE,SegSet,load_model,top_tiles\nOUT=Path(os.environ[\'E8C_OUT\'])\nCOST_CONFIG={\'schema\':\'dense-cost-and-k-scenarios-v2\',\'image_shape\':[1,3,512,1024],\n             \'dtype\':\'float16 segmenters, float32 optional adapter core\',\'images\':5,\'repeats_per_image\':5,\n             \'warmup\':3,\'component_repeats\':5,\'kv_repeats\':10,\n             \'flop_convention\':\'fvcore: one multiply-add counts as one operation\',\n             \'no_sparse_runtime_implemented\':True}\n\ndef write_json(path,data):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    p=path.with_suffix(path.suffix+\'.part\');p.write_text(json.dumps(data,indent=2));p.replace(path)\n\ndef save_csv(path,rows):\n    if not rows:return\n    fields=list(dict.fromkeys(k for r in rows for k in r))\n    with path.open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)\n\nclass Logits(nn.Module):\n    def __init__(self,model):super().__init__();self.model=model\n    def forward(self,x):return self.model(pixel_values=x).logits\n\n@torch.inference_mode()\ndef measured(fn,inputs,repeats=5,warmup=3):\n    for _ in range(warmup):fn(inputs[0])\n    torch.cuda.synchronize()\n    torch.cuda.reset_peak_memory_stats()\n    gpu=[];wall=[]\n    for x in inputs:\n        for _ in range(repeats):\n            torch.cuda.synchronize()\n            a=torch.cuda.Event(enable_timing=True);b=torch.cuda.Event(enable_timing=True)\n            t=time.perf_counter();a.record();result=fn(x);b.record();torch.cuda.synchronize()\n            wall.append(1000*(time.perf_counter()-t));gpu.append(a.elapsed_time(b))\n            del result\n    return {\'samples\':len(gpu),\'gpu_mean_ms\':float(np.mean(gpu)),\n            \'gpu_p95_ms\':float(np.percentile(gpu,95)),\n            \'wall_mean_ms\':float(np.mean(wall)),\'wall_p95_ms\':float(np.percentile(wall,95)),\n            \'peak_allocated_mib\':torch.cuda.max_memory_allocated()/2**20}\n\n\ndef components(model):\n    enc=model.segformer.encoder\n    groups=[]\n    for st in range(4):\n        prefix=f\'stage{st+1}\'\n        groups.append((prefix+\'_patch\',enc.patch_embeddings[st],enc.patch_embeddings[st]))\n        groups.append((prefix+\'_blocks\',enc.block[st][0],enc.block[st][-1]))\n        groups.append((prefix+\'_norm\',enc.layer_norm[st],enc.layer_norm[st]))\n    groups.append((\'decode_head\',model.decode_head,model.decode_head))\n    return groups\n\n@torch.inference_mode()\ndef timed_components(model,x):\n    # Non-overlapping groups; separate instrumented passes, not the latency benchmark above.\n    pending={};pairs=defaultdict(list);handles=[]\n    def begin(name):\n        def hook(module,args):\n            event=torch.cuda.Event(enable_timing=True);event.record();pending[name]=event\n        return hook\n    def end(name):\n        def hook(module,args,out):\n            event=torch.cuda.Event(enable_timing=True);event.record();pairs[name].append((pending.pop(name),event))\n        return hook\n    for name,first,last in components(model):\n        handles.append(first.register_forward_pre_hook(begin(name)))\n        handles.append(last.register_forward_hook(end(name)))\n    try:\n        for _ in range(COST_CONFIG[\'component_repeats\']):model(pixel_values=x)\n        torch.cuda.synchronize()\n        return [{\'component\':name,\'gpu_mean_ms\':float(np.mean([a.elapsed_time(b) for a,b in events])),\n                 \'instrumented\':True} for name,events in pairs.items()]\n    finally:\n        for h in handles:h.remove()\n\n@torch.inference_mode()\ndef flop_counts(model,x):\n    from fvcore.nn import FlopCountAnalysis\n    wrapped=Logits(model).eval()\n    f=FlopCountAnalysis(wrapped,(x,)).unsupported_ops_warnings(False).uncalled_modules_warnings(False).tracer_warnings(\'none\')\n    total=float(f.total());by=f.by_module()\n    rows=[]\n    for st in range(4):\n        for label,path in [(\'patch\',f\'model.segformer.encoder.patch_embeddings.{st}\'),\n                           (\'blocks\',f\'model.segformer.encoder.block.{st}\'),\n                           (\'norm\',f\'model.segformer.encoder.layer_norm.{st}\')]:\n            value=float(by.get(path,0))\n            rows.append({\'component\':f\'stage{st+1}_{label}\',\'counted_gops_fvcore\':value/1e9,\n                         \'share_of_counted_total\':value/max(total,1)})\n    value=float(by.get(\'model.decode_head\',0))\n    rows.append({\'component\':\'decode_head\',\'counted_gops_fvcore\':value/1e9,\'share_of_counted_total\':value/max(total,1)})\n    return {\'counted_total_gops\':total/1e9,\'components\':rows,\n            \'by_operator\':{k:float(v) for k,v in f.by_operator().items()},\n            \'unsupported_ops\':dict(f.unsupported_ops()),\n            \'uncalled_modules\':sorted(f.uncalled_modules()),\n            \'note\':\'Counted operations only; explicit unsupported ops are not silently treated as measured FLOPs.\'}\n\n\ndef dominant_macs(block,height,width):\n    a=block.attention.self\n    q=a.query;k=a.key;v=a.value;o=block.attention.output.dense\n    assert q.in_features==k.in_features==v.in_features\n    assert q.out_features==k.out_features==v.out_features\n    assert q.in_features==q.out_features,\'Unpruned equal-width attention expected\'\n    n=height*width;d=q.in_features\n    if a.sr_ratio>1:\n        sr=a.sr\n        kh,kw=sr.kernel_size;sh,sw=sr.stride;ph,pw=sr.padding;dh,dw=sr.dilation\n        hk=(height+2*ph-dh*(kh-1)-1)//sh+1\n        wk=(width+2*pw-dw*(kw-1)-1)//sw+1\n        nk=hk*wk\n        sr_mac=nk*sr.out_channels*(sr.in_channels//sr.groups)*kh*kw\n    else:nk=n;sr_mac=0\n    f=block.mlp;conv=f.dwconv.dwconv\n    return {\'sr\':int(sr_mac),\'kv\':int(nk*(k.in_features*k.out_features+v.in_features*v.out_features)),\n            \'q\':int(n*q.in_features*q.out_features),\n            \'attention_products\':int(2*n*nk*q.out_features),\n            \'attention_output\':int(n*o.in_features*o.out_features),\n            \'ffn_dense1\':int(n*f.dense1.in_features*f.dense1.out_features),\n            \'ffn_dwconv\':int(n*conv.out_channels*(conv.in_channels//conv.groups)*conv.kernel_size[0]*conv.kernel_size[1]),\n            \'ffn_dense2\':int(n*f.dense2.in_features*f.dense2.out_features)}\n\n\ndef halo_fractions(tile_masks):\n    native=torch.from_numpy(np.asarray(tile_masks)).float().repeat_interleave(TILE//16,1).repeat_interleave(TILE//16,2)\n    expanded=F.max_pool2d(native[:,None],kernel_size=3,stride=1,padding=1)[:,0]\n    return float(native.mean()),float(expanded.mean())\n\n\ndef sparse_macs(parts,rho,halo):\n    fixed=parts[\'sr\']+parts[\'kv\']\n    query_halo=parts[\'q\']+parts[\'attention_products\']+parts[\'attention_output\']+parts[\'ffn_dense1\']\n    selected=parts[\'ffn_dwconv\']+parts[\'ffn_dense2\']\n    return {\'fixed_sr_kv\':fixed,\'full_dominant_macs\':sum(parts.values()),\n            \'optimistic_no_halo_macs\':fixed+rho*(query_halo+selected),\n            \'halo_aware_macs\':fixed+halo*query_halo+rho*selected}\n\n@torch.inference_mode()\ndef collect_stage3(model,x):\n    layers=model.segformer.encoder.block[2]\n    inputs=[None]*len(layers);shape={};handles=[]\n    def pre(i):\n        def hook(module,args):inputs[i]=args[0].detach();shape[\'height\']=int(args[1]);shape[\'width\']=int(args[2])\n        return hook\n    for i,block in enumerate(layers):handles.append(block.register_forward_pre_hook(pre(i)))\n    try:model(pixel_values=x)\n    finally:\n        for h in handles:h.remove()\n    return inputs,shape\n\n\ndef kv_path(block,x,height,width):\n    # Precisely the full-context normalization/SR/K/V path, no queries or FFN.\n    x=block.layer_norm_1(x);a=block.attention.self\n    if a.sr_ratio>1:\n        x=a.sr(x.transpose(1,2).reshape(x.shape[0],x.shape[2],height,width))\n        x=a.layer_norm(x.flatten(2).transpose(1,2))\n    return a.key(x),a.value(x)\n\n\n@torch.inference_mode()\ndef optional_adapter():\n    path=RESULTS.parent/\'e9_adapter_v1/adapter_best.pt\'\n    if not path.exists():return {\'available\':False,\'note\':\'E9 best checkpoint not found; no adapter latency measured\'}\n    checkpoint=torch.load(path,map_location=\'cpu\',weights_only=True)\n    state=checkpoint[\'model\'];in_channels=state[\'net.0.weight\'].shape[1]\n    hidden=state[\'net.0.weight\'].shape[0];out_channels=state[\'net.2.weight\'].shape[0]\n    class Adapter(nn.Module):\n        def __init__(self):\n            super().__init__();self.net=nn.Sequential(nn.Conv2d(in_channels,hidden,1),nn.GELU(),nn.Conv2d(hidden,out_channels,1))\n        def forward(self,x):return self.net(x)\n    model=Adapter().cuda().float().eval().requires_grad_(False);model.load_state_dict(state)\n    x=torch.zeros((1,in_channels,SIZE[0]//16,SIZE[1]//16),device=\'cuda\',dtype=torch.float32)\n    latency=measured(model,[x],repeats=20,warmup=3)\n    n=x.shape[-2]*x.shape[-1]\n    count=n*(in_channels*hidden+hidden*out_channels)\n    result={\'available\':True,\'checkpoint_sha256\':hashlib.sha256(path.read_bytes()).hexdigest(),\n            \'parameters\':sum(p.numel() for p in model.parameters()),\'core_gmac\':count/1e9,\'latency\':latency,\n            \'note\':\'MLP core only, fp32 as E9. Input on GPU; normalization/denormalization, B0 and mask handling excluded.\'}\n    del model,x;gc.collect();torch.cuda.empty_cache()\n    return result\n\n@torch.inference_mode()\ndef measure_dense_and_initial_scenarios():\n    assert torch.cuda.is_available(),\'Select GPU runtime\'\n    OUT.mkdir(parents=True,exist_ok=True)\n    config=dict(COST_CONFIG,source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest())\n    config_path=OUT/\'cost_config.json\'\n    if config_path.exists():assert json.loads(config_path.read_text())==config,\'Changed code/config: use new E8C_OUT\'\n    else:write_json(config_path,config)\n    import transformers\n    hardware={\'gpu\':torch.cuda.get_device_name(0),\'gpu_memory_bytes\':torch.cuda.get_device_properties(0).total_memory,\n              \'torch\':torch.__version__,\'cuda\':torch.version.cuda,\'transformers\':transformers.__version__,\n              \'python\':platform.python_version(),\'tf32_matmul\':torch.backends.cuda.matmul.allow_tf32}\n    data=SegSet(\'cs_test\');indices=np.linspace(0,len(data)-1,min(COST_CONFIG[\'images\'],len(data)),dtype=int)\n    imgs=[data[int(i)][0][None].cuda().half() for i in indices]\n    report={\'hardware\':hardware,\'config\':config,\'sample_names\':[data.items[int(i)][0].stem for i in indices],\n            \'models\':{},\'note\':\'P95 is over this small microbenchmark, not dataset latency P95.\'}\n    latency_rows=[];component_rows=[]\n    for name in [\'b0\',\'b5\']:\n        print(\'Cost:\',name,flush=True)\n        model=load_model(name).requires_grad_(False)\n        raw=measured(lambda x:model(pixel_values=x).logits,imgs,COST_CONFIG[\'repeats_per_image\'],COST_CONFIG[\'warmup\'])\n        def mask_forward(x):\n            logits=model(pixel_values=x).logits\n            return F.interpolate(logits.float(),SIZE,mode=\'bilinear\',align_corners=False).argmax(1)[0].byte().cpu().numpy()\n        masks=measured(mask_forward,imgs,COST_CONFIG[\'repeats_per_image\'],COST_CONFIG[\'warmup\'])\n        parts=timed_components(model,imgs[0])\n        flop=flop_counts(model,imgs[0])\n        report[\'models\'][name]={\'parameters\':sum(p.numel() for p in model.parameters()),\n                                \'forward_gpu_resident\':raw,\'to_cpu_mask\':masks,\'timed_components\':parts,\'fvcore\':flop}\n        latency_rows.extend([dict(model=name,path=\'forward_gpu_resident\',**raw),dict(model=name,path=\'to_cpu_mask\',**masks)])\n        for row in flop[\'components\']:\n            timer=next(t for t in parts if t[\'component\']==row[\'component\'])\n            component_rows.append(dict(model=name,**row,instrumented_gpu_mean_ms=timer[\'gpu_mean_ms\']))\n        if name==\'b5\':\n            inputs,shape=collect_stage3(model,imgs[0]);height=shape[\'height\'];width=shape[\'width\']\n            layers=model.segformer.encoder.block[2]\n            dparts=[dominant_macs(block,height,width) for block in layers]\n            summed={k:sum(p[k] for p in dparts) for k in dparts[0]}\n            def full_kv(_):\n                # Calls each layer\'s distinct weights on its actual dense-pass input.\n                for block,x in zip(layers,inputs):out=kv_path(block,x,height,width)\n                return out\n            kv=measured(full_kv,[None],repeats=COST_CONFIG[\'kv_repeats\'],warmup=3)\n            report[\'stage3\']={\'height\':height,\'width\':width,\'blocks\':len(layers),\'dominant_macs\':summed,\n                \'full_sr_kv_fraction_of_dominant_macs\':(summed[\'sr\']+summed[\'kv\'])/sum(summed.values()),\n                \'full_context_kv_path_all_blocks\':kv,\n                \'kv_note\':\'Standalone microbenchmark including LN1/SR/LN/K/V; not additive to measured forward latency. Masks/queries/FFN excluded.\'}\n            del inputs\n        write_json(OUT/\'cost_measured.json\',report)\n        del model;gc.collect();torch.cuda.empty_cache()\n    report[\'adapter\']=optional_adapter()\n    write_json(OUT/\'cost_measured.json\',report)\n    save_csv(OUT/\'cost_latency.csv\',latency_rows)\n    save_csv(OUT/\'cost_components.csv\',component_rows)\n    # Arithmetic scenarios: full B5 early/later stages remain; sparse stage3 not implemented.\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:\n        def rank(a):return a.ravel().argsort().argsort().reshape(a.shape)\n        score=rank(z[\'cs_test__entropy__mean\'])+rank(z[\'cs_test__knn__mean\'])\n    b0_count=report[\'models\'][\'b0\'][\'fvcore\'][\'counted_total_gops\']\n    b5_count=report[\'models\'][\'b5\'][\'fvcore\'][\'counted_total_gops\']\n    s3_count=next(r[\'counted_gops_fvcore\'] for r in report[\'models\'][\'b5\'][\'fvcore\'][\'components\'] if r[\'component\']==\'stage3_blocks\')\n    fixed_other=b5_count-s3_count\n    adapter_gmac=report[\'adapter\'].get(\'core_gmac\')\n    scenarios=[]\n    for budget in [.1,.2]:\n        tile_mask=top_tiles(score,budget)\n        selected,halo=halo_fractions(tile_mask)\n        active=float(tile_mask.reshape(len(tile_mask),-1).any(1).mean())\n        calc=sparse_macs(report[\'stage3\'][\'dominant_macs\'],selected,halo)\n        row={\'budget\':budget,\'selected_native_fraction\':selected,\'halo_query_fraction\':halo,\n             \'active_frame_fraction\':active,\n             \'stage3_full_dominant_gmac\':calc[\'full_dominant_macs\']/1e9,\n             \'stage3_fixed_sr_kv_gmac\':calc[\'fixed_sr_kv\']/1e9,\n             \'stage3_optimistic_no_halo_gmac\':calc[\'optimistic_no_halo_macs\']/1e9,\n             \'stage3_halo_aware_gmac\':calc[\'halo_aware_macs\']/1e9,\n             \'b5_non_stage3_counted_gops\':fixed_other,\'b0_full_counted_gops\':b0_count,\n             \'adapter_core_gmac\':adapter_gmac,\n             \'assumptions\':\'Planning arithmetic only: full B0, B5 outside stage3 unchanged, full SR/KV every block; sparse query/FFN; 1-token FFN halo. No sparse latency implementation.\'}\n        if adapter_gmac is not None:\n            row[\'hybrid_planning_count_halo_aware\']=b0_count+adapter_gmac+fixed_other+calc[\'halo_aware_macs\']/1e9\n            row[\'hybrid_planning_ratio_to_counted_b5\']=row[\'hybrid_planning_count_halo_aware\']/b5_count\n            row[\'hybrid_planning_count_skip_empty_frames\']=b0_count+active*(adapter_gmac+fixed_other)+(calc[\'halo_aware_macs\']-(1-active)*calc[\'fixed_sr_kv\'])/1e9\n            row[\'hybrid_planning_ratio_skip_empty_frames\']=row[\'hybrid_planning_count_skip_empty_frames\']/b5_count\n        scenarios.append(row)\n    write_json(OUT/\'cost_scenarios.json\',{\'rows\':scenarios,\n        \'warning\':\'Hybrid approximation mixes fvcore supported-op counts for dense parts with dominant MACs for sparse stage3. It omits stage3 layernorm/softmax/GELU/residual/masking/indexing/assembly, adapter normalization, router, transfers and unhandled operations. It is not measured FLOPs or a latency/speedup prediction.\'})\n    save_csv(OUT/\'cost_scenarios.csv\',scenarios)\n    frame_curve=frame_baseline(data,score,b0_count,b5_count)\n    save_csv(OUT/\'cached_frame_baseline.csv\',frame_curve)\n    print(json.dumps(report[\'stage3\'],indent=2),flush=True)\n    print(\'Measured latency:\',latency_rows,flush=True)\n    print(\'Arithmetic scenarios:\',scenarios,flush=True)\n\ndef frame_baseline(data,tile_score,b0_count,b5_count):\n    # Existing predictions only. Quality/control-budget curve, no live cascade timing.\n    from common import confusion,miou\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:\n        methods={\'raw_entropy\':z[\'cs_test__entropy__mean\'].mean((1,2)),\n                 \'rank_mean\':tile_score.mean((1,2))}\n    records=[]\n    for image,_ in data.items:\n        with np.load(CACHE/\'cs_test\'/(image.stem+\'.npz\')) as z:\n            records.append((confusion(z[\'pred_b0\'],z[\'gt\']),confusion(z[\'pred_b5\'],z[\'gt\'])))\n    records=np.asarray(records);rows=[];n=len(records)\n    for method,score in methods.items():\n        order=np.argsort(-score,kind=\'stable\')\n        for budget in [0.,.05,.1,.2,.3,.5,.75,1.]:\n            m=np.zeros(n,bool);m[order[:round(budget*n)]]=True\n            c=np.where(m[:,None,None],records[:,1],records[:,0]).sum(0)\n            fraction=float(m.mean())\n            rows.append({\'method\':method,\'requested_frame_fraction\':budget,\'actual_frame_fraction\':fraction,\n                         \'cached_miou\':float(miou(c)),\n                         \'counted_gops_b0_plus_selected_full_b5\':b0_count+fraction*b5_count,\n                         \'note\':\'Transductive domain-wide frame ranking, cached predictions; not measured cascade latency.\'})\n    return rows\n\n\n\n\nOLD_DENSE_SOURCE_SHA256 = \'faa4d2ef339df630853dd1327627d170aadefaf88c0e74276168d2a975c0b071\'\n\ndef reuse_measured():\n    import shutil\n    origin=Path(os.environ.get(\'E8C_MEASURED_REUSE\',str(RESULTS.parent/\'e8c_cost_v1\')))\n    required=[\'cost_measured.json\',\'cost_latency.csv\',\'cost_components.csv\',\'cost.py\']\n    if not all((origin/name).is_file() for name in required):return False\n    report=json.loads((origin/\'cost_measured.json\').read_text())\n    source_hash=hashlib.sha256((origin/\'cost.py\').read_bytes()).hexdigest()\n    if source_hash!=OLD_DENSE_SOURCE_SHA256 or report[\'config\'].get(\'source_sha256\')!=source_hash:\n        print(\'Не используем старый профиль: не совпала версия кода.\',flush=True);return False\n    hw=report[\'hardware\']\n    if hw[\'gpu\']!=torch.cuda.get_device_name(0) or hw[\'torch\']!=torch.__version__ or hw[\'cuda\']!=torch.version.cuda:\n        print(\'Не используем старый профиль: другое GPU/окружение.\',flush=True);return False\n    if hw.get(\'tf32_matmul\')!=torch.backends.cuda.matmul.allow_tf32:return False\n    if report[\'config\'].get(\'image_shape\')!=COST_CONFIG[\'image_shape\']:return False\n    if not all(name in report[\'models\'] for name in [\'b0\',\'b5\']):return False\n    if \'stage3\' not in report:return False\n    report[\'reuse\']={\'source\':str(origin),\'historical_measurements\':True,\n                     \'not_measured_again\':True,\'original_code_sha256\':source_hash}\n    write_json(OUT/\'cost_measured.json\',report)\n    for name in [\'cost_latency.csv\',\'cost_components.csv\']:\n        shutil.copy2(origin/name,OUT/name)\n    write_json(OUT/\'cost_config.json\',dict(COST_CONFIG,source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                                         measured_reused_from=str(origin)))\n    print(\'Используем уже измеренный профиль:\',origin,flush=True)\n    return True\n\n\ndef k_scenarios(report):\n    with np.load(CACHE/\'e8_tiles_cityscapes.npz\') as z:\n        def rank(a):return a.ravel().argsort().argsort().reshape(a.shape)\n        score=rank(z[\'cs_test__entropy__mean\'])+rank(z[\'cs_test__knn__mean\'])\n    b0_count=report[\'models\'][\'b0\'][\'fvcore\'][\'counted_total_gops\']\n    b5_count=report[\'models\'][\'b5\'][\'fvcore\'][\'counted_total_gops\']\n    s3_count=next(r[\'counted_gops_fvcore\'] for r in report[\'models\'][\'b5\'][\'fvcore\'][\'components\'] if r[\'component\']==\'stage3_blocks\')\n    fixed_other=b5_count-s3_count\n    n=report[\'stage3\'][\'height\']*report[\'stage3\'][\'width\']\n    scenarios=[]\n    for k in [1,2,4,8]:\n        c_out=320*(k+1)  # k distinct contexts PLUS final restoration, not k outputs total.\n        adapter_gmac=n*(160*128+128*c_out)/1e9\n        parameters=160*128+128+128*c_out+c_out\n        for budget in [.1,.2]:\n            tile_mask=top_tiles(score,budget)\n            rho,halo=halo_fractions(tile_mask)\n            active=float(tile_mask.reshape(len(tile_mask),-1).any(1).mean())\n            calc=sparse_macs(report[\'stage3\'][\'dominant_macs\'],rho,halo)\n            total=b0_count+adapter_gmac+fixed_other+calc[\'halo_aware_macs\']/1e9\n            skipped=b0_count+active*(adapter_gmac+fixed_other)+(calc[\'halo_aware_macs\']-(1-active)*calc[\'fixed_sr_kv\'])/1e9\n            scenarios.append({\'k\':k,\'adapter_output_channels\':c_out,\'adapter_parameters_scenario\':parameters,\n              \'adapter_core_gmac_scenario\':adapter_gmac,\'budget\':budget,\n              \'selected_native_fraction\':rho,\'halo_query_fraction\':halo,\'active_frame_fraction\':active,\n              \'stage3_fixed_sr_kv_gmac\':calc[\'fixed_sr_kv\']/1e9,\n              \'stage3_halo_aware_gmac\':calc[\'halo_aware_macs\']/1e9,\n              \'b5_non_stage3_counted_gops\':fixed_other,\'b0_full_counted_gops\':b0_count,\n              \'hybrid_planning_count_halo_aware\':total,\'hybrid_planning_ratio_to_counted_b5\':total/b5_count,\n              \'hybrid_planning_count_skip_empty_frames\':skipped,\n              \'hybrid_planning_ratio_skip_empty_frames\':skipped/b5_count,\n              \'status\':\'unimplemented arithmetic scenario; not measured cascade performance\'})\n    write_json(OUT/\'cost_scenarios.json\',{\'rows\':scenarios,\n        \'warning\':\'Dense costs and sparse arithmetic are separate. All early/later B5 parts remain full-frame. Halo is accounted for in stage3. k>1 adapters are hypothetical; only k=1 adapter has trained weights. No sparse latency is measured; do not attach E9 quality to a hypothetical regional-early-stage cost.\'})\n    save_csv(OUT/\'cost_scenarios.csv\',scenarios)\n    frame_rows=frame_baseline(SegSet(\'cs_test\'),score,b0_count,b5_count)\n    save_csv(OUT/\'cached_frame_baseline.csv\',frame_rows)\n    for row in scenarios:print(row,flush=True)\n\n\ndef main():\n    assert torch.cuda.is_available(),\'Select GPU runtime\'\n    OUT.mkdir(parents=True,exist_ok=True)\n    if not reuse_measured():measure_dense_and_initial_scenarios()\n    report=json.loads((OUT/\'cost_measured.json\').read_text())\n    k_scenarios(report)\nif __name__==\'__main__\':main()\n'}
for name,text in NEW_SOURCES.items():
    for path in [EXP/name,OUT/name]:
        if path.exists():
            assert path.read_text()==text, f'Другая версия {path}; выбери новую папку OUT для новой версии.'
        else:path.write_text(text)
print('Код готов')


In [ ]:
# 8. Запуск с логом и сообщениями каждые 30 секунд.
def run_next(script,stage=None):
    label=Path(script).stem+('_'+stage if stage else '')
    log_path=OUT/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    command=[sys.executable,'-u',str(EXP/script)]
    if stage:command.append(stage)
    print('Запуск:',label,'| Лог:',log_path,flush=True)
    start=time.monotonic()
    with log_path.open('w') as log:
        process=subprocess.Popen(command,cwd=EXP,env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    tail=log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{label}: {(time.monotonic()-start)/60:.1f} мин | '+' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(log_path.read_text(errors='replace')[-14000:])
    assert status==0, f'Ошибка {status}, смотри {log_path}'
    print('Готово за',round((time.monotonic()-start)/60,1),'мин')


## Отдельный отчёт №1: измеренные затраты

Сначала используем уже полученный профиль из `e8c_cost_v1`, если совпадают код и окружение. В `cost_measured.json` сохраняются исходные результаты и происхождение; `reuse` отмечает, что измерения не повторялись.

Если повтор нужен: batch=1, 512×1024, fp16, 5 кадров × 5 повторов, 3 warmup. Считается raw forward и путь до CPU-маски, отдельно компоненты модели, SR/K/V и ядро адаптера E9. Ядро адаптера измеряется в fp32, как в E9. Роутер, нормализация/денормализация, чтение диска и перенос входа сюда не входят.

fvcore считает только поддерживаемые операции (один multiply-add = одна операция); список неподдержанных операций сохраняется. P95 микробенчмарка не является p95 всего датасета. Отдельные компонентные измерения нельзя механически складывать в доказанную латентность нового каскада.


In [ ]:
# 9. Прежние измерения (если совместимы) + новые сценарии для k=1/2/4/8.
run_next('e10_cost_k.py')


In [ ]:
# 10. Быстрый просмотр вычислительной цены.
import pandas as pd
from IPython.display import display
print('Измеренная латентность:')
display(pd.read_csv(OUT/'cost_latency.csv'))
print('Компоненты моделей:')
display(pd.read_csv(OUT/'cost_components.csv'))
measured=json.loads((OUT/'cost_measured.json').read_text())
print('Использованы прежние измерения:',measured.get('reuse',False))
print('Неучтённые операции fvcore:')
for name,result in measured['models'].items():print(name,result['fvcore']['unsupported_ops'])
print('Standalone K/V path:',measured['stage3']['full_context_kv_path_all_blocks'])
print('Доля SR+K/V в доминирующих MAC stage 3:',measured['stage3']['full_sr_kv_fraction_of_dominant_macs'])
print('Адаптер:',measured['adapter'])
print('Расчётные сценарии — НЕ измеренное ускорение:')
display(pd.read_csv(OUT/'cost_scenarios.csv'))


## Отдельный отчёт №2: сценарная стоимость будущей реализации

Для каждого блока доминирующие MAC разделены на:

\[
C_{\mathrm{fixed}}=C_{\mathrm{SR}}+C_K+C_V,
\]

\[
C_{\mathrm{halo}}=C_Q+C_{QK}+C_{AV}+C_{\mathrm{out}}+C_{\mathrm{FFN1}},
\qquad
C_{\mathrm{selected}}=C_{\mathrm{DWConv}}+C_{\mathrm{FFN2}}.
\]

Если доля выбранных токенов — \(\rho\), а их расширения на один токен — \(\rho_h\), сценарная стоимость:

\[
C_{\mathrm{sparse}}\approx C_{\mathrm{fixed}}
+\rho_h C_{\mathrm{halo}}+\rho C_{\mathrm{selected}}.
\]

Расширение нужно потому, что depthwise-свёртка FFN 3×3 читает соседей: для точного вычисления FFN выбранного токена нужны также attention-результаты и первая проекция FFN соседей. Halo посчитан на фактических масках `rank` из готового кэша.

Показаны два режима планирования: B5 ранние/поздние стадии на всех кадрах и пропуск B5 на кадрах без единого выбранного тайла. Второй использует фактическую долю активных кадров.

Итоговый `hybrid_planning_*` приближённый: поддерживаемые fvcore-операции плотных частей смешаны с доминирующими MAC разреженной stage 3. В нём отсутствуют часть нормализаций/softmax/GELU, индексирование, сборка, роутер и передачи. **Не переводите это число напрямую в миллисекунды или доказанное ускорение.**

Также сохранён `cached_frame_baseline.csv`: качество покадровой маршрутизации B0→полный B5 при разных долях кадров, по тем же готовым предсказаниям. Это контроль качества/арифметического бюджета, не живой замер каскада. Сравнения с окнами E5b здесь нет: для него нужны отдельные кроп-предсказания и замеры.

Метод подсчёта: [fvcore documentation](https://detectron2.readthedocs.io/en/stable/modules/fvcore.html).


**Многовыходный адаптер:** для k контекстов нужен ещё отдельный final для восстановления выхода. При 320 каналах размер выхода = `320 × (k+1)`: 640, 960, 1600, 2880 каналов для k=1/2/4/8. Стоимость расширенной головы считается аналитически; реально обучен и измерен только нынешний адаптер k=1.

Нельзя приписывать качество E9 гипотетической стоимости региональных ранних стадий или stage 4: они в E9 были полнокадровыми. В этих сценариях все части B5 вне блоков stage 3 сохранены полнокадровыми. Веса новых k>1 адаптеров ещё не существуют.


## E8c: состояния из середины сегментов

Используются те же `rank`, маски 10%/20% и `cs_test` из 233 кадров. Выход вне выбранных тайлов восстанавливается настоящим final B5.

| k | Входы блоков, используемые как контекст |
|---|---|
| 1 | 20 — контроль E9/E8b |
| 2 | 10, 30 |
| 4 | 5, 15, 25, 35 |
| 8 | 2, 7, 12, 17, 22, 27, 32, 37 |

Это отличается от первого E8c с точками 0/10/20/30. На каждый кадр — один обычный проход для записи состояний и восемь маскированных проходов: четыре k × два бюджета.

**Строгие контроли:** запись не меняет обычные логиты; all-true для каждого k совпадает с обычным проходом; k=40 на половине тайлов воспроизводит предсказание ВСЕГО кадра и близкие логиты. В конце k=1 сравнивается с настоящим E9/E8b JSON; расхождение больше 0.01 п.п. останавливает публикацию отчёта и bootstrap.

Каждые 10 кадров сохраняются промежуточные матрицы и число завершённых кадров. При перезапуске отсчёт продолжается от последнего сохранённого кадра. Результаты ещё не означают, что адаптер сможет предсказать все эти состояния.


In [ ]:
# 11. E8c k-sweep. Без обучения.
run_next('e8c_k_sweep.py','quality')


In [ ]:
# 12. Парный bootstrap: 1000 повторов, k против k=1 и full_tiles.
run_next('e8c_k_sweep.py','bootstrap')


In [ ]:
# 13. Итог и архив без тяжёлого кэша.
import zipfile
print('Качество:')
display(pd.read_csv(OUT/'e8c_quality.csv'))
print('Bootstrap:')
display(pd.DataFrame(json.loads((OUT/'e8c_bootstrap.json').read_text())))
print('Покадровый контроль из кэша:')
display(pd.read_csv(OUT/'cached_frame_baseline.csv'))
archive=OUT/'e8c_k_sweep_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=archive and not p.name.endswith('.part'):z.write(p,p.name)
    for p in [SAVE/'run_config.json',RESULTS/'e0_cityscapes.json',RESULTS/'e8_cityscapes.json',
              SAVE/'e8b_v1/e8b_cityscapes.json',SAVE/'e9_adapter_v1/training_complete.json',EXP/'common.py']:
        if p.is_file():z.write(p,'reference/'+p.name)
print('Пришли:',archive)
# Скачать:
# from google.colab import files
# files.download(str(archive))


## Как интерпретировать

- Сначала убедись, что `sanity.json` и `cs_test_reference_check.json` прошли проверки. При MISMATCH скрипт останавливается; не подбирать допуски, чтобы принять результат.
- Сравни `gain_vs_b0_pp`, `delta_vs_k1_pp`, потерю относительно `full_tiles` и интервалы. Несколько расписаний оцениваются на уже знакомом cs_test — вывод исследовательский, а интервалы номинальные, без поправки на множественные сравнения.
- Улучшение k=4/8 обосновывает отдельную проверку многовыходного адаптера, но не доказывает возможность разреженного ускорения. Отсутствие улучшения не доказывает необходимость LoRA.
- Измеренную цену и аналитические сценарии держим раздельно. Не переводим расчётные отношения в доказанный speedup.

При сбросе VM выполни notebook с начала: восстановится E8-кэш, качество продолжится с последнего checkpoint каждые 10 кадров. Готовый итоговый домен пропускается. Веса B0/B5/E9 не изменяются.
